# Retreivial

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

### Configuration

In [2]:
db_path = "db/chroma_db"
embedding_model_path = r"D:\\AI Projects\\Applied AI Engineering\\Projects\\Trained Models\\HuggingFaceEmbeddings Models\\all-mpnet-base-v2"

### embedding model

In [3]:
embedding_model = HuggingFaceEmbeddings(
    model=embedding_model_path,
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
# connect to db

db = Chroma(
    collection_name="weekly_reports",  
    persist_directory=db_path,
    embedding_function=embedding_model
)

In [5]:
collection_count = db._collection.count()
collection_count

7

In [6]:
retreiver = db.as_retriever(search_kwargs = {"k":3})

In [9]:
user_question = "What work was done for the  Weekly Report Management System?"
relevant_docs = retreiver.invoke(user_question)

In [10]:
for i, doc in enumerate(relevant_docs):
    print(doc.page_content[:100])
    print("-"*100)

Project: Weekly Report Management System
        User: johndoe
        Week: 2026-07-13 to 2026-07-1
----------------------------------------------------------------------------------------------------
Project: Hospital Management System
        User: johndoe
        Week: 2026-07-13 to 2026-07-19
   
----------------------------------------------------------------------------------------------------
Project: Banking Transaction Analytics
        User: johndoe
        Week: 2026-07-13 to 2026-07-19

----------------------------------------------------------------------------------------------------


In [11]:
relevant_docs

[Document(id='c05076ff-b63a-4f8b-a950-f9dc548deffb', metadata={'project_id': 2, 'report_id': 4, 'user_id': 'ab2a05fa-6e18-47ee-9574-40b0a60fa650'}, page_content='Project: Weekly Report Management System\n        User: johndoe\n        Week: 2026-07-13 to 2026-07-19\n        \n        Tasks Completed:\n        Implemented report approval workflow, added report status tracking, and fixed authentication issues.\n        \n        Tasks Planned:\n        Develop manager analytics dashboard and notification system.\n        \n        Blockers:\n        Awaiting UI mockups for dashboard.\n        \n        Hours Worked: 41\n        Status: DRAFT'),
 Document(id='319e8177-3649-475e-aead-6c4c8143256f', metadata={'project_id': 5, 'user_id': 'ab2a05fa-6e18-47ee-9574-40b0a60fa650', 'report_id': 7}, page_content='Project: Hospital Management System\n        User: johndoe\n        Week: 2026-07-13 to 2026-07-19\n        \n        Tasks Completed:\n        Developed patient appointment reminders, im

In [12]:
### LLM

In [13]:
combined_docs = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])
combined_docs

'- Project: Weekly Report Management System\n        User: johndoe\n        Week: 2026-07-13 to 2026-07-19\n        \n        Tasks Completed:\n        Implemented report approval workflow, added report status tracking, and fixed authentication issues.\n        \n        Tasks Planned:\n        Develop manager analytics dashboard and notification system.\n        \n        Blockers:\n        Awaiting UI mockups for dashboard.\n        \n        Hours Worked: 41\n        Status: DRAFT\n- Project: Hospital Management System\n        User: johndoe\n        Week: 2026-07-13 to 2026-07-19\n        \n        Tasks Completed:\n        Developed patient appointment reminders, improved doctor scheduling, and optimized database queries.\n        \n        Tasks Planned:\n        Implement billing module and medical report export.\n        \n        Blockers:\n        Integration with SMS provider is pending.\n        \n        Hours Worked: 42\n        Status: DRAFT\n- Project: Banking Transacti

In [14]:
prompt = f"""
    User message :
        {user_question}

    Documents : 
        {combined_docs}

    Instruction : 
        Generate a helpful response for Weekly Report Management System using 'Documents'. 
        If you can't find helpfull answer in docs say I'm wasn't able to find any information please contact support.
"""

In [15]:
llm = ChatOllama(
    model='llama3'
)

In [19]:
print(llm)

metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.7'}} output_version=None model='llama3'


In [20]:
response = llm.invoke(prompt).content

In [21]:
print(response)

According to the provided documents, significant work was done on the Weekly Report Management System this week. The tasks completed include:

* Implemented report approval workflow
* Added report status tracking
* Fixed authentication issues

These improvements aim to enhance the system's functionality and usability. Additionally, planning is underway for further development, including the creation of a manager analytics dashboard and notification system.

However, some work remains pending, specifically awaiting UI mockups for the dashboard. This may impact the progress of the planned tasks.

Overall, 41 hours were spent working on this project this week, with an overall status of DRAFT.
